In [1]:
from ts_toolkit.pipeline.runner import TsPipeline
from ts_toolkit.pipeline.utils import load_config
import logging
from tqdm.notebook import tqdm
import pandas as pd
from dotenv import load_dotenv
import os
import numpy as np
from gluonts.model.forecast import QuantileForecast
import utilsforecast.processing as ufp
from gluonts.model import Forecast
from multivar_timeseries.evaluation.metrics import MAR, MBR

# auto reload imports in jupyter
%load_ext autoreload
%autoreload 2

/home/roc/01_projects/01_timeseries/timeseries-research/.venv/lib/python3.10/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


In [2]:
load_dotenv()
print(GIFT_EVAL_PATH := os.getenv("GIFT_EVAL"))
print(REPO_ROOT := os.getenv("REPO_ROOT"))

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

/home/roc/01_projects/01_timeseries/timeseries-research/data/gift_eval
/home/roc/01_projects/01_timeseries/timeseries-research


In [3]:
logging.basicConfig(level=logging.INFO)

# --- Load configuration ---
# config_path = "./configs/experiments/simpletime.yaml"
# config_path = "../configs/experiments/simpletime_sample.yaml"
config_path = "configs/experiments/personal/debug_all_missing.yaml"
# config_path = "./configs/experiments/gifteval.yaml"

config = load_config(config_path)

# --- Set up the evaluation pipeline ---
pipeline = TsPipeline(config)
# pipeline.run(skip_existing=False)

# pipeline.get_results()


Loading base config from <REPO_ROOT>/configs/experiments/personal/debug_all_missing.yaml...


INFO:root:   - Results will be saved to: /home/roc/01_projects/01_timeseries/timeseries-research/results/debug_all_missing_20251021-145901/ToTo-GiftEval-sample
INFO:root:   - Datasets will be loaded from: /home/roc/01_projects/01_timeseries/timeseries-research/data/gift_eval



In [4]:
config.experiment_name

'debug_all_missing_20251021-145901'

# Investigate

## Get Forecasts

In [5]:
import pandas as pd
import logging
from tqdm.notebook import tqdm
from ts_toolkit.models.ensembles.median import MedianEnsemble
from ts_toolkit.models import MULTIVARIATE_MODEL_LIST
from ts_toolkit.pipeline.gluonts_predictor import GluonTSPredictor
from ts_toolkit.pipeline.eval import Evaluator
from ts_toolkit.models import MODEL_REGISTRY, get_model
import os


In [6]:
dataset_cfg = pipeline.config.datasets[0]
to_univariate = pipeline.config.evaluation.to_univariate


In [7]:
evaluator = Evaluator(
    dataset_name=dataset_cfg.get("name"),
    term=dataset_cfg.get("term", "short"),
    to_univariate=to_univariate,
    # Create a subdirectory for each model's results
    output_path=f"{pipeline.output_path}/{pipeline.forecaster.alias}",
    storage_path=pipeline.storage_path,
    config=pipeline.config
)


In [8]:
dataset = evaluator.dataset.test_data.input
dataset

InputDataset(test_data=TestData(dataset=<gluonts.transform._base.TransformedDataset object at 0x7a519e122920>, splitter=OffsetSplitter(offset=-96), prediction_length=48, windows=2, distance=48, max_history=None))

In [9]:
h = pipeline.predictor.h or dataset.test_data.prediction_length
freq = pipeline.predictor.freq
print(h, freq, pipeline.predictor.batch_size)

48 None 1024


In [10]:
evaluator.forecasts = pipeline.predictor.predict(dataset)

100%|██████████| 16/16 [00:17<00:00,  1.07s/it]
/home/roc/01_projects/01_timeseries/timeseries-research/src/ts_toolkit/models/ensembles/median.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fcst_df[self.alias] = _fcst_df[model_cols].median(axis=1)
/home/roc/01_projects/01_timeseries/timeseries-research/src/ts_toolkit/models/ensembles/median.py:139: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fcst_df[q_col] = _fcst_df[models_q_cols].median(axis=1)
/home/roc/01_projects/01_timeseries/timeseries-resea

In [11]:
evaluator.forecasts.__len__(), [len(x.mean) for x in evaluator.forecasts][0]

(2000, 48)

In [12]:
batch = []
for _, entry in enumerate(dataset):
    batch.append(entry)

# So batch is w * n_var * T
len(batch), [wind['target'].shape for wind in batch]

(2000,
 [(624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (

In [13]:
# Extract all values for each variate across all windows
variates = batch[0]['target'].shape[0]  # number of variates
windows = len(batch)

# var_i: all values of variate i from all windows concatenated
var_dict = {}
for i in range(variates):
    var_dict[f'var_{i+1}'] = [batch[w]['target'][i].flatten() for w in range(windows)]
    var_dict[f'var_{i+1}'] = np.concatenate(var_dict[f'var_{i+1}'])

# var_i_j: all values of variate i from window j (j=1-based)
for i in range(variates):
    for j in range(windows):
        var_dict[f'var_{i+1}_{j+1}'] = batch[j]['target'][i].flatten()

# Example: var_1 contains all values of variate 1 from all windows
#          var_1_1 contains all values of variate 1 from window 1

# Check if the first 2376 values of all windows are the same for each variate
same_across_windows = {}
for i in range(variates):
    key = f'var_{i+1}_1'
    reference = var_dict[key][:2376]
    all_equal = True
    for j in range(2, windows+1):
        compare_key = f'var_{i+1}_{j}'
        if not np.array_equal(reference, var_dict[compare_key][:2376]):
            all_equal = False
            break
    same_across_windows[f'var_{i+1}'] = all_equal

same_across_windows

{'var_1': False,
 'var_2': False,
 'var_3': False,
 'var_4': False,
 'var_5': False,
 'var_6': False,
 'var_7': False,
 'var_8': False,
 'var_9': False,
 'var_10': False,
 'var_11': False,
 'var_12': False,
 'var_13': False,
 'var_14': False,
 'var_15': False,
 'var_16': False,
 'var_17': False,
 'var_18': False,
 'var_19': False,
 'var_20': False,
 'var_21': False,
 'var_22': False,
 'var_23': False,
 'var_24': False,
 'var_25': False,
 'var_26': False,
 'var_27': False,
 'var_28': False,
 'var_29': False,
 'var_30': False,
 'var_31': False,
 'var_32': False,
 'var_33': False,
 'var_34': False,
 'var_35': False,
 'var_36': False,
 'var_37': False,
 'var_38': False,
 'var_39': False,
 'var_40': False,
 'var_41': False,
 'var_42': False,
 'var_43': False,
 'var_44': False,
 'var_45': False,
 'var_46': False,
 'var_47': False,
 'var_48': False,
 'var_49': False,
 'var_50': False,
 'var_51': False,
 'var_52': False,
 'var_53': False,
 'var_54': False,
 'var_55': False,
 'var_56': False,
 

### Findings:
> 1. evaluator.dataset.test_data.input, which converts to the batch input, is in the shape of (n_windows, n_var, n_time_step), Each window and var contains all values of the previous window + 48 new time steps, which is the `h`
> 2. df is just all n_windows concatenated together vertically, each window with the same starting ds. So the number of rows of df = sum of all time steps in the (test) dataset. Multivariates are converted into y_i columns
> 3. predict_df takes df to run MedianEnsemble.forecast(), and returns fcst_df in the shape of (n_windows x n_var x h,  2 (unique_id, ds) + 1 (median) + 9 (quantiles)) or (6 x 7 x 48, 12)
> 4. Last bit of predict_df wrangles fcst_df to the shape of (n_windows x n_var, h) or (6 * 7, 48)

In [14]:
df = pipeline.predictor._gluonts_dataset_to_df(batch)
df.shape

(1024000, 3)

In [15]:
metadata = pipeline.predictor.multivar_metadata if len(pipeline.predictor.target_cols) > 1 else pipeline.predictor.univar_metadata

In [16]:
fcst_df = pipeline.predictor.forecaster.forecast(
            df=df,
            h=h,
            freq=freq,
            level=pipeline.predictor.level,
            quantiles=pipeline.predictor.quantiles,
        )

100%|██████████| 32/32 [00:35<00:00,  1.10s/it]
/home/roc/01_projects/01_timeseries/timeseries-research/src/ts_toolkit/models/ensembles/median.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fcst_df[self.alias] = _fcst_df[model_cols].median(axis=1)
/home/roc/01_projects/01_timeseries/timeseries-research/src/ts_toolkit/models/ensembles/median.py:139: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fcst_df[q_col] = _fcst_df[models_q_cols].median(axis=1)
/home/roc/01_projects/01_timeseries/timeseries-resea

In [17]:
fcst_df = fcst_df.set_index("unique_id")
fcsts: list[Forecast] = []
for uid, metadata_uid in tqdm(metadata.items(), total=len(metadata)):
    fcst_df_uid = fcst_df.loc[uid]
    forecast_arrays = ufp.value_cols_to_numpy(
        df=fcst_df_uid,
        id_col="unique_id",
        time_col="ds",
        target_col=None,
    )
    forecast_keys = ["mean"]
    if pipeline.predictor.quantiles is not None:
        forecast_keys += [f"{q}" for q in pipeline.predictor.quantiles]
    q_fcst = QuantileForecast(
        forecast_arrays=forecast_arrays.T,
        forecast_keys=forecast_keys,
        item_id=metadata_uid["item_id"],
        start_date=metadata_uid["fcst_start"],
    )
    fcsts.append(q_fcst)


  0%|          | 0/2000 [00:00<?, ?it/s]

In [18]:
fcsts.__len__(), fcsts[0].quantile(0.5).__len__()

(2000, 48)

In [19]:
fcsts_50qtl = [fcst.quantile(0.5) for fcst in fcsts]

## Evaluate

### Evaluation Pipeline Summary

1. **Batching Data and Forecasts**  
   The original data and forecasts are passed to `gluonts.model.evaluation.evaluate_forecasts_raw`, where `data.input`, `data.label`, and forecasts are parsed and grouped into batches using a batcher.

2. **Batch Wrapping with ChainMap**  
   For each batch, the input, label, and forecasts are combined into a `ChainMap` object called `data_batch`, which is basically a merged dictionary for convenient access and is then passed to each metric (e.g., MAE, MSE) for evaluation. Here is why we need to modify the source code of `gluonts`, because in the original package `input` is only used to derive seasonality. The original `input` time series are not included in the final `data_batch`. 

3. **Metric Calculation Process**  
   - Each metric is instantiated as an evaluator, and the calculation is performed via `evaluator.update(data_batch)`.
   - The evaluator has two main components:
     - `stat`: Defines the error calculation (e.g., `absolute_error` for MAE). A `partial` wrapper is used to fix parameters such as `forecast_type='0.5'`.
     - `aggregate`: Typically a `Mean()` class with a specified axis for aggregation.
   - Under the hood, `evaluator.update` executes `aggregate.step(stat(data))`:
     - `stat(data)` computes the error for the batch (e.g., `absolute_error(data_batch, forecast_type='0.5')`).
     - `aggregate.step()` processes the error and updates:
       - `aggregate.partial_result`: The sum of all errors in the batch.
       - `aggregate.get()`: The mean error, calculated as `aggregate.partial_result / aggregate.n`.

4. **Aggregating Results Across Batches**  
   As the evaluation iterates through all batches, the evaluator accumulates the total error (`partial_result`) and the total number of time steps (`n`). The final average error across all batches is obtained via `aggregate.get()`.

### Notes
1. For multivar data, metrics are calculated with all vars flattened into one - aggregated over all time steps. Make sense when the number of steps among vars are largely different

### Preproc

In [20]:
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)

from gluonts.itertools import batcher

from gluonts.model.evaluation import _get_data_batch
from toolz import first

In [21]:
test_data = evaluator.dataset.test_data
# test_data = evaluator.dataset.univar_test_data
axis = None
metrics = [
    MAE(),
]

batch_size=512 # ?

# forecasts = fcsts_reshaped
forecasts = fcsts


EVALUATOR = Evaluator(
    dataset_name=dataset_cfg.get("name"),
    term=dataset_cfg.get("term", "short"),
    to_univariate=to_univariate,
    # Create a subdirectory for each model's results
    output_path=f"{pipeline.output_path}/{pipeline.forecaster.alias}",
    storage_path=pipeline.storage_path,
    config=pipeline.config
)

seasonality = EVALUATOR.seasonality
mask_invalid_label = True
allow_nan_forecast = False

In [22]:
label_ndim = first(test_data.label)["target"].ndim

assert label_ndim in [1, 2]

if axis is None:
    axis = tuple(range(label_ndim + 1))
if isinstance(axis, int):
    axis = (axis,)

assert all(ax in range(3) for ax in axis)

evaluators = {}
evaluator_metric = MAE()(axis=axis)
evaluators[evaluator_metric.name] = evaluator_metric


index_data = []

# TODO: Here the returned batch size is actually just one, within which there are several windows. 
# We may want to align the batcher and the windows, or remove batcher altogether.
input_batches = batcher(test_data.input, batch_size=batch_size)
label_batches = batcher(test_data.label, batch_size=batch_size)
forecast_batches = batcher(forecasts, batch_size=batch_size)

# pbar = tqdm()

input_batch, label_batch, forecast_batch = first(zip(input_batches, label_batches, forecast_batches))

if 0 not in axis:
    index_data.extend(
        [
            (forecast.item_id, forecast.start_date)
            for forecast in forecast_batch
        ]
    )


data_batch = _get_data_batch(
            input_batch,
            label_batch,
            forecast_batch,
            seasonality=seasonality,
            mask_invalid_label=mask_invalid_label,
            allow_nan_forecast=allow_nan_forecast,
        )

In [23]:
print(data_batch['input'].__len__(), data_batch['label'][0].__len__())
data_batch['input'][481]

512 48


array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, na

In [24]:
print(data_batch['label'].shape)
data_batch['label'][481]

(512, 48)


masked_array(data=[--, --, --, --, --, --, --, --, --, --, --,
                   28.869863510131836, 26.821664810180664,
                   27.146774291992188, 26.008886337280273,
                   26.496553421020508, 26.659109115600586,
                   27.309329986572266, 26.984220504760742,
                   26.496553421020508, 26.17144203186035,
                   26.17144203186035, 26.33399772644043,
                   26.33399772644043, 26.496553421020508,
                   26.496553421020508, 26.496553421020508,
                   26.33399772644043, 26.496553421020508,
                   27.146774291992188, 26.17144203186035,
                   25.846330642700195, 26.984220504760742,
                   26.659109115600586, 26.33399772644043,
                   26.008886337280273, 26.984220504760742,
                   26.659109115600586, 25.683774948120117,
                   26.496553421020508, 25.846330642700195,
                   26.17144203186035, 25.683774948120117,
 

In [25]:
print(data_batch['0.5'].shape)
data_batch['0.5'][481]

(512, 48)


array([ 5.54495418e-05, -4.71946914e-05, -7.08205334e-05, -3.12625081e-04,
       -2.68445801e-05,  1.12636801e-04,  6.18811318e-05, -3.32561176e-05,
        3.24182620e-05, -7.68109312e-05, -3.25409492e-05, -6.37172488e-05,
       -9.50966787e-05, -1.12270827e-05,  2.35983534e-05, -9.33645788e-05,
       -2.32760358e-04, -1.75859663e-04,  1.24872284e-04,  1.17013231e-04,
       -4.07847438e-05, -3.27020025e-05, -1.21609395e-04,  9.72885464e-05,
        2.95422687e-05, -2.21370738e-05,  1.80438110e-05,  9.64874107e-06,
       -4.27082850e-05, -1.03253886e-04, -7.36624497e-05, -2.60262284e-04,
       -2.17783687e-04, -1.02234451e-04,  1.36833172e-04, -5.95839338e-05,
        9.60577599e-05, -8.50792640e-05,  4.97538786e-05,  1.37777097e-04,
       -1.08035456e-04, -1.11799149e-04,  3.70919406e-05, -9.48802772e-05,
       -8.32334626e-05, -1.23013069e-05, -1.15110786e-04, -9.01222375e-05])